<a href="https://colab.research.google.com/github/Navya40869/edge-idps-colab/blob/main/pipeline/04_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import time
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ==========================================
# 1. MOUNT DRIVE & LOAD ARTIFACTS
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/Edge-IDPS-Data/processed_data/"

scaler = joblib.load(save_path + "scaler.pkl")
label_encoder = joblib.load(save_path + "label_encoder.pkl")
X_test = np.load(save_path + "X_test.npy")

num_features = X_test.shape[1]
num_classes = len(label_encoder.classes_)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-define Model Architecture to load weights
class EdgeIDPSClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(EdgeIDPSClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# Load PyTorch Weights
model = EdgeIDPSClassifier(num_features, num_classes).to(device)
model.load_state_dict(torch.load(save_path + "edge_idps_model.pth", map_location=device))
model.eval()

print("All pipeline artifacts successfully loaded into memory!")

# ==========================================
# 2. REAL-TIME STREAMING SIMULATION LOOP
# ==========================================
output_log_path = save_path + "live_stream_output.csv"

# Initialize or reset the live streaming log file with column headers
log_headers = ["timestamp", "packet_id", "predicted_class", "confidence", "alert_status"]
pd.DataFrame(columns=log_headers).to_csv(output_log_path, index=False)

print("\n🚀 Starting Real-Time Network Packet Streaming Pipeline...")
print(f"Logging outputs continuously to: {output_log_path}\n")

# Stream 50 simulated packet samples sequentially
num_packets_to_stream = 50

for packet_id in range(1, num_packets_to_stream + 1):
    # Pick a random sample from the test set (simulating an incoming network frame)
    sample_index = np.random.randint(0, len(X_test))
    raw_packet = X_test[sample_index].reshape(1, -1)

    # Process through PyTorch model
    tensor_packet = torch.tensor(raw_packet, dtype=torch.float32).to(device)

    with torch.no_grad():
        outputs = model(tensor_packet)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, dim=1)

    predicted_label = label_encoder.inverse_transform([predicted_idx.item()])[0]
    conf_score = confidence.item() * 100

    # Determine alert status
    alert_status = "BENIGN" if str(predicted_label).lower() in ["benign", "normal", "0"] else "CRITICAL ATTACK"
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

    # Log details
    new_entry = pd.DataFrame([{
        "timestamp": timestamp,
        "packet_id": f"PKT_{packet_id:04d}",
        "predicted_class": predicted_label,
        "confidence": f"{conf_score:.2f}%",
        "alert_status": alert_status
    }])

    # Append to output log for Member 4's dashboard
    new_entry.to_csv(output_log_path, mode='a', header=False, index=False)

    print(f"[{timestamp}] Packet {packet_id:02d} | Class: {predicted_label} | Conf: {conf_score:.1f}% | Status: {alert_status}")

    # Simulate network packet delay (e.g., 0.5 sec)
    time.sleep(0.5)

print("\nStreaming loop completed! Log file updated for Member 4.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All pipeline artifacts successfully loaded into memory!

🚀 Starting Real-Time Network Packet Streaming Pipeline...
Logging outputs continuously to: /content/drive/MyDrive/Edge-IDPS-Data/processed_data/live_stream_output.csv

[2026-07-28 08:58:11] Packet 01 | Class: 13 | Conf: 98.6% | Status: CRITICAL ATTACK
[2026-07-28 08:58:11] Packet 02 | Class: 6 | Conf: 100.0% | Status: CRITICAL ATTACK
[2026-07-28 08:58:12] Packet 03 | Class: 12 | Conf: 50.2% | Status: CRITICAL ATTACK
[2026-07-28 08:58:12] Packet 04 | Class: 11 | Conf: 56.0% | Status: CRITICAL ATTACK
[2026-07-28 08:58:13] Packet 05 | Class: 9 | Conf: 100.0% | Status: CRITICAL ATTACK
[2026-07-28 08:58:14] Packet 06 | Class: 9 | Conf: 100.0% | Status: CRITICAL ATTACK
[2026-07-28 08:58:14] Packet 07 | Class: 8 | Conf: 100.0% | Status: CRITICAL ATTACK
[2026-07-28 08:58:15] Packet 08 | Class: 14 | Conf: 98.2% 

In [3]:
import pandas as pd
df = pd.read_csv(save_path + "live_stream_output.csv")
print(f"Log file successfully verified! Total logged packets: {len(df)}")
df.head()

Log file successfully verified! Total logged packets: 50


,timestamp,packet_id,predicted_class,confidence,alert_status
0,2026-07-28 08:58:11,PKT_0001,13,98.65%,CRITICAL ATTACK
1,2026-07-28 08:58:11,PKT_0002,6,100.00%,CRITICAL ATTACK
2,2026-07-28 08:58:12,PKT_0003,12,50.20%,CRITICAL ATTACK
3,2026-07-28 08:58:12,PKT_0004,11,55.96%,CRITICAL ATTACK
4,2026-07-28 08:58:13,PKT_0005,9,100.00%,CRITICAL ATTACK
